In [1]:
%cd /content

!git clone https://github.com/abderrahim-elouardi/small-agentic-rag-benchmark.git


/content
Cloning into 'small-agentic-rag-benchmark'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 119 (delta 30), reused 115 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 17.21 MiB | 16.12 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [8]:
%cd small-agentic-rag-benchmark

%ls





/content/small-agentic-rag-benchmark
app/
artifacts/
configs/
data/
docs/
experiments/
guide_preprocessing_mini_transformer.pdf
LICENSE
models/
notebooks/
ps_scripts/
pyproject.toml
rapport_mini_transformer_travail_2026-10-09.pdf
README.md
reports/
scripts/
Small-Agentic-RAG-Quantized-Evaluation-Bench-Guide.pdf
src/
temperature_top_k_mini_gpt.pdf
tests/
uv.lock


In [10]:
import torch

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print(
        "VRAM totale :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2,
        ),
        "Go",
    )

PyTorch : 2.11.0+cu130
CUDA disponible : True
GPU : Tesla T4
VRAM totale : 14.56 Go


In [1]:
%pip install -U transformers accelerate bitsandbytes

In [2]:
import torch
import transformers
import bitsandbytes as bnb
import accelerate

print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("BitsAndBytes :", bnb.__version__)
print("Accelerate :", accelerate.__version__)
print("GPU :", torch.cuda.get_device_name(0))

PyTorch : 2.11.0+cu130
Transformers : 5.20.1
BitsAndBytes : 0.50.2
Accelerate : 1.15.0
GPU : Tesla T4


In [5]:
import os

print("Répertoire courant :", os.getcwd())
print("Contenu de /content :", os.listdir("/content"))

Répertoire courant : /content
Contenu de /content : ['.config', 'small-agentic-rag-benchmark', 'sample_data']


In [6]:
%cd /content/small-agentic-rag-benchmark

import os

print("Contenu du projet :", os.listdir())
print("Contenu de src :", os.listdir("src"))

/content/small-agentic-rag-benchmark
Contenu du projet : ['models', '.gitignore', 'ps_scripts', 'uv.lock', 'temperature_top_k_mini_gpt.pdf', 'src', 'configs', 'tests', 'guide_preprocessing_mini_transformer.pdf', 'reports', '.python-version', 'README.md', '.env.example', 'experiments', '.git', 'app', 'pyproject.toml', 'LICENSE', 'artifacts', 'rapport_mini_transformer_travail_2026-10-09.pdf', 'docs', 'data', 'notebooks', 'Small-Agentic-RAG-Quantized-Evaluation-Bench-Guide.pdf', 'scripts']
Contenu de src : ['small_agentic_rag']


In [7]:
import sys

sys.path.insert(
    0,
    "/content/small-agentic-rag-benchmark/src"
)

from small_agentic_rag.llm.model_loader import ModelLoader
from small_agentic_rag.llm.quantization import QuantizationConfig

print("Imports réussis !")

quant_config = QuantizationConfig(
    method="bitsandbytes_4bit",
    compute_dtype="float16",
    quant_type="nf4",
    use_double_quant=True,
)

loader = ModelLoader(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    device="cuda",
    quantization_config=quant_config,
)

tokenizer, model = loader.load()

print("Modèle chargé :", model.config._name_or_path)
print("Quantification :", quant_config.method)
print("Type des paramètres :", next(model.parameters()).dtype)
print("VRAM allouée :", round(torch.cuda.memory_allocated() / 1024**3, 2), "Go")
print("VRAM réservée :", round(torch.cuda.memory_reserved() / 1024**3, 2), "Go")

Imports réussis !


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modèle chargé : Qwen/Qwen2.5-0.5B-Instruct
Quantification : bitsandbytes_4bit
Type des paramètres : torch.float16
VRAM allouée : 0.43 Go
VRAM réservée : 0.45 Go


In [8]:
import bitsandbytes as bnb

linear4bit_count = 0
other_linear_count = 0

for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        linear4bit_count += 1
    elif isinstance(module, torch.nn.Linear):
        other_linear_count += 1

print("Couches Linear4bit :", linear4bit_count)
print("Couches Linear classiques :", other_linear_count)
print("Type de quantification configuré :", quant_config.method)

Couches Linear4bit : 168
Couches Linear classiques : 1
Type de quantification configuré : bitsandbytes_4bit


In [17]:
from small_agentic_rag.llm.inference import TextGenerator

generator = TextGenerator(
    model=model,
    tokenizer=tokenizer,
    device="cuda"
)

result = generator.generate(
    "Explique brièvement ce qu'est un modèle de langage.",
    max_new_tokens=80,
)

print(result)

Un modèle de langage est une structure qui permet à différents types d'interactions (par exemple, des conversations entre deux personnes ou des messages sur internet) de communiquer efficacement avec l'intégralité du public. Il sert à définir les règles et les actions que le système doit suivre pour produire une communication appropriée. Par exemple, si vous avez un modèle


In [29]:
!git pull origin main


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 366 bytes | 366.00 KiB/s, done.
From https://github.com/abderrahim-elouardi/small-agentic-rag-benchmark
 * branch            main       -> FETCH_HEAD
   8a86379..8d3c8eb  main       -> origin/main
Updating 8a86379..8d3c8eb
Fast-forward
 scripts/benchmark_quantization.py | 3 ++-
 1 file changed, 2 insertions(+), 1 deletion(-)


In [21]:
%cd /content/small-agentic-rag-benchmark
!PYTHONPATH=src python scripts/benchmark_quantization.py

/content/small-agentic-rag-benchmark
GPU: Tesla T4
VRAM: 14.56 GiB
Model: Qwen/Qwen2.5-0.5B-Instruct
Measured runs: 10

Benchmark: FP16
Loading weights: 100% 290/290 [00:00<00:00, 394.13it/s]
timestamp_utc: 2026-10-19T16:44:50.124913+00:00
model: Qwen/Qwen2.5-0.5B-Instruct
gpu: Tesla T4
configuration: FP16
torch_version: 2.11.0+cu130
transformers_version: 5.20.1
bitsandbytes_version: 0.50.2
max_new_tokens: 80
warmup_runs: 2
benchmark_runs: 10
load_seconds: 3.280165606999617
memory_after_load_gb: 0.9278974533081055
peak_load_memory_gb: 0.9278974533081055
peak_inference_memory_gb: 0.9403724670410156
mean_latency_seconds: 2.412356214100009
median_latency_seconds: 2.410709988999997
mean_generated_tokens: 80
mean_tokens_per_second: 33.17266141496345
median_tokens_per_second: 33.18525066800022

Benchmark: 4-bit NF4
Loading weights: 100% 290/290 [00:01<00:00, 265.68it/s]
timestamp_utc: 2026-10-19T16:45:35.549653+00:00
model: Qwen/Qwen2.5-0.5B-Instruct
gpu: Tesla T4
configuration: 4-bit NF4
to

In [25]:
!git add .
!git commit -m"docs: LLM quantization benchmark results"



Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@3b15c8126593.(none)')


## **chargement de model plus grand**

In [28]:

import sys
sys.path.insert(0, "/content/small-agentic-rag-benchmark/src")

import torch
from small_agentic_rag.llm.model_loader import ModelLoader
from small_agentic_rag.llm.quantization import QuantizationConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

config = QuantizationConfig(
    method="bitsandbytes_4bit",
    compute_dtype="float16",
    quant_type="nf4",
    use_double_quant=True,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

loader = ModelLoader(
    model_name=MODEL_NAME,
    device="cuda",
    torch_dtype="float16",
    quantization_config=config,
)

tokenizer, model = loader.load()

print("Modèle chargé :", MODEL_NAME)
print("Mémoire GPU utilisée :",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GiB")
print("Pic de mémoire GPU :",
      round(torch.cuda.max_memory_allocated() / 1024**3, 3), "GiB")

prompt = "Explain briefly how a transformer language model works."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modèle chargé : Qwen/Qwen2.5-1.5B-Instruct
Mémoire GPU utilisée : 1.509 GiB
Pic de mémoire GPU : 1.533 GiB
Explain briefly how a transformer language model works. A transformer is an advanced neural network architecture that can be used for various tasks such as natural language processing, computer vision, and speech recognition. It consists of multiple layers of self-attention mechanisms that allow the model to attend to different parts of its input sequence at each time step. The attention mechanism allows the model to focus on relevant information from different positions in the input sequence, which helps it capture long


In [31]:
%cd /content/small-agentic-rag-benchmark
!uv sync
!uv pip install -e .

/content/small-agentic-rag-benchmark
Resolved 88 packages in 1ms
Checked 84 packages in 2ms
Using Python 3.13.15 environment at: /usr
Resolved 83 packages in 693ms
Prepared 1 package in 822ms
Uninstalled 12 packages in 816ms
Installed 13 packages in 221ms
 - cuda-toolkit==13.0.2
 + cuda-toolkit==13.0.3.0
 - datasets==4.8.5
 + datasets==5.1.0
 - matplotlib==3.10.0
 + matplotlib==3.11.2
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.24.0.43
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.30.7
 - pyarrow==23.0.1
 + pyarrow==26.0.0
 + small-agentic-rag-benchmark==0.1.0 (from file:///content/small-agentic-rag-benchmark)
 - tokenizers==0.23.2
 + tokenizers==0.23.3
 - torch==2.11.0+cu130
 + torch==2.14.1
 - torchvision==0.26.0+cu130
 + torchvision==0.29.1
 - triton==3.6.0
 + triton==3.8.0


In [33]:
%cd /content/small-agentic-rag-benchmark

!pwd
!ls -la
!find src -maxdepth 4 -type f | head -60
!cat pyproject.toml

/content/small-agentic-rag-benchmark
/content/small-agentic-rag-benchmark
total 1412
drwxr-xr-x 18 root root   4096 Oct 19 16:59 .
drwxr-xr-x  1 root root   4096 Oct 19 16:04 ..
drwxr-xr-x  2 root root   4096 Oct 19 16:04 app
drwxr-xr-x  6 root root   4096 Oct 19 16:04 artifacts
drwxr-xr-x  6 root root   4096 Oct 19 16:04 configs
drwxr-xr-x  5 root root   4096 Oct 19 16:04 data
drwxr-xr-x  3 root root   4096 Oct 19 16:04 docs
-rw-r--r--  1 root root      0 Oct 19 16:04 .env.example
drwxr-xr-x  5 root root   4096 Oct 19 16:04 experiments
drwxr-xr-x  8 root root   4096 Oct 19 16:58 .git
-rw-r--r--  1 root root   1669 Oct 19 16:04 .gitignore
-rw-r--r--  1 root root  81921 Oct 19 16:04 guide_preprocessing_mini_transformer.pdf
-rw-r--r--  1 root root      0 Oct 19 16:04 LICENSE
drwxr-xr-x  2 root root   4096 Oct 19 16:04 models
drwxr-xr-x  2 root root   4096 Oct 19 16:04 notebooks
drwxr-xr-x  2 root root   4096 Oct 19 16:04 ps_scripts
-rw-r--r--  1 root root    485 Oct 19 16:04 pyproject.to

In [34]:
%cd /content/small-agentic-rag-benchmark

import sys
sys.path.insert(0, "/content/small-agentic-rag-benchmark/src")

from small_agentic_rag.llm.model_loader import ModelLoader

print("Import réussi")

/content/small-agentic-rag-benchmark
Import réussi


In [37]:
%cd /content/small-agentic-rag-benchmark

import os
import sys

project_root = "/content/small-agentic-rag-benchmark"
src_path = os.path.join(project_root, "src")

print("Dossier courant :", os.getcwd())
print("Le dossier src existe :", os.path.isdir(src_path))
print("Le package existe :", os.path.isdir(
    os.path.join(src_path, "small_agentic_rag")
))
print("Contenu de src :", os.listdir(src_path) if os.path.isdir(src_path) else "Absent")

sys.path.insert(0, src_path)

from small_agentic_rag.llm.model_loader import ModelLoader

print("Import réussi !")

/content/small-agentic-rag-benchmark
Dossier courant : /content/small-agentic-rag-benchmark
Le dossier src existe : True
Le package existe : True
Contenu de src : ['small_agentic_rag_benchmark.egg-info', 'small_agentic_rag']
Import réussi !


In [38]:
from small_agentic_rag.llm.model_loader import ModelLoader

print("Import de ModelLoader réussi !")

Import de ModelLoader réussi !


In [39]:
from pathlib import Path

project_dir = Path("/content/small-agentic-rag-benchmark")

for path in sorted((project_dir / "src" / "small_agentic_rag").rglob("*.py")):
    print(path.relative_to(project_dir))

src/small_agentic_rag/__init__.py
src/small_agentic_rag/agents/__init__.py
src/small_agentic_rag/agents/guardrails.py
src/small_agentic_rag/agents/prompts.py
src/small_agentic_rag/agents/router.py
src/small_agentic_rag/agents/state.py
src/small_agentic_rag/agents/tool_registry.py
src/small_agentic_rag/agents/tools/__init__.py
src/small_agentic_rag/agents/tools/calculator_tool.py
src/small_agentic_rag/agents/tools/rag_tool.py
src/small_agentic_rag/agents/workflow.py
src/small_agentic_rag/evaluation/__init__.py
src/small_agentic_rag/evaluation/benchmark_runner.py
src/small_agentic_rag/evaluation/generation_metrics.py
src/small_agentic_rag/evaluation/judge.py
src/small_agentic_rag/evaluation/latency_metrics.py
src/small_agentic_rag/evaluation/memory_metrics.py
src/small_agentic_rag/evaluation/report_generator.py
src/small_agentic_rag/evaluation/retrieval_metrics.py
src/small_agentic_rag/llm/__init__.py
src/small_agentic_rag/llm/generation_config.py
src/small_agentic_rag/llm/inference.py
s

In [35]:
!uv run python scripts/benchmark_quantization.py

Traceback (most recent call last):
  File "/content/small-agentic-rag-benchmark/scripts/benchmark_quantization.py", line 16, in <module>
    from small_agentic_rag.llm.model_loader import ModelLoader
ModuleNotFoundError: No module named 'small_agentic_rag'
